In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px
import warnings
warnings.filterwarnings('ignore')


In [2]:
try:
    df=pd.read_csv('insurance.csv')
    print('Data Loaded Successfully')
except:
    print('Dataset not Found')

Data Loaded Successfully


In [3]:
df

,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520
...,...,...,...,...,...,...,...
1333,50,male,30.970,3,no,northwest,10600.54830
1334,18,female,31.920,0,no,northeast,2205.98080
1335,18,female,36.850,0,no,southeast,1629.83350
1336,21,female,25.800,0,no,southwest,2007.94500


In [4]:
df.isnull().sum().sort_values(ascending=False)

age         0
sex         0
bmi         0
children    0
smoker      0
region      0
charges     0
dtype: int64

In [5]:
df['charges'] = df['charges'].round(0).astype('Int64')

In [6]:
df

,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16885
1,18,male,33.770,1,no,southeast,1726
2,28,male,33.000,3,no,southeast,4449
3,33,male,22.705,0,no,northwest,21984
4,32,male,28.880,0,no,northwest,3867
...,...,...,...,...,...,...,...
1333,50,male,30.970,3,no,northwest,10601
1334,18,female,31.920,0,no,northeast,2206
1335,18,female,36.850,0,no,southeast,1630
1336,21,female,25.800,0,no,southwest,2008


In [7]:
px.box(df,x='age')

In [8]:
px.box(df,x='charges')

In [9]:
px.box(df,x='bmi')

In [10]:
X=df.drop('charges',axis=1)
y=df['charges']

In [11]:
from sklearn.compose import ColumnTransformer,make_column_selector as selector
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder,RobustScaler,OneHotEncoder

num=Pipeline([
    ('num',RobustScaler())
])
cat=Pipeline([
    ('cat',OneHotEncoder(handle_unknown='ignore'))
])
preprocessor=ColumnTransformer([
    ('nums',num,selector(dtype_include=np.number)),
    ('cats',cat,selector(dtype_include=['object','category']))
])
preprocessor

,transformers,"[('nums', ...), ('cats', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True
,force_int_remainder_cols,'deprecated'
,with_centering,True
,with_scaling,True
,quantile_range,"(25.0, ...)"


In [12]:
from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split,GridSearchCV,KFold,RandomizedSearchCV

In [13]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)
model=Pipeline([
    ('preprocessing',preprocessor),
    ('model',XGBRegressor())
])
model

,steps,"[('preprocessing', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('nums', ...), ('cats', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [14]:
param_grid={
    'model__n_estimators':[100,500,1000],
    'model__max_depth':[1,3,5,10],
    'model__learning_rate':[0.3]
}
cv=GridSearchCV(
    param_grid=param_grid,
    estimator=model,
    scoring='neg_mean_squared_error',
    cv=15,
    verbose=1
)

cv.fit(X_train,y_train)
y_pred=cv.predict(X_test)

Fitting 15 folds for each of 12 candidates, totalling 180 fits


In [15]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_absolute_percentage_error,
    mean_squared_error,
    root_mean_squared_error,
    r2_score,
    d2_absolute_error_score,
    d2_pinball_score
)


mae = mean_absolute_error(y_test, y_pred)
mape = mean_absolute_percentage_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = root_mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
d2_abs = d2_absolute_error_score(y_test, y_pred)
d2_pin = d2_pinball_score(y_test, y_pred)

print(f"MAE:   {mae:.4f}")
print(f"MAPE:  {mape:.4f}")
print(f"MSE:   {mse:.4f}")
print(f"RMSE:  {rmse:.4f}")
print(f"R²:    {r2:.4f}")
print(f"D² Absolute Error Score: {d2_abs:.4f}")
print(f"D² Pinball Score:        {d2_pin:.4f}")


MAE:   2533.4170
MAPE:  0.3050
MSE:   19669218.0000
RMSE:  4434.9990
R²:    0.8733
D² Absolute Error Score: 0.7051
D² Pinball Score:        0.7051
